In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, brier_score_loss

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

RANDOM_STATE = 42
DATA_DIR = Path("data/interim")

# -----------------------------
# load data
# -----------------------------
m_train = pd.read_csv(DATA_DIR / "m_matchups_train_2014_2024.csv")
m_valid = pd.read_csv(DATA_DIR / "m_matchups_valid_2025.csv")
m_full  = pd.read_csv(DATA_DIR / "m_matchups_full_train_2014_2025.csv")
m_pred  = pd.read_csv(DATA_DIR / "m_matchups_stage2_2026.csv")

w_train = pd.read_csv(DATA_DIR / "w_matchups_train_2014_2024.csv")
w_valid = pd.read_csv(DATA_DIR / "w_matchups_valid_2025.csv")
w_full  = pd.read_csv(DATA_DIR / "w_matchups_full_train_2014_2025.csv")
w_pred  = pd.read_csv(DATA_DIR / "w_matchups_stage2_2026.csv")

print("M train:", m_train.shape, "M valid:", m_valid.shape, "M pred:", m_pred.shape)
print("W train:", w_train.shape, "W valid:", w_valid.shape, "W pred:", w_pred.shape)

M train: (669, 270) M valid: (67, 270) M pred: (66430, 268)
W train: (642, 237) W valid: (67, 237) W pred: (65703, 235)


In [2]:
def get_feature_columns(train_df, valid_df, pred_df):
    common_cols = sorted(set(train_df.columns) & set(valid_df.columns) & set(pred_df.columns))

    # tylko numeryczne Diff_* i Same_*
    feat_cols = []
    for c in common_cols:
        if c in ["Target", "ID", "Sex", "Season", "DayNum", "NumOT", "TeamIDLow", "TeamIDHigh"]:
            continue
        if c.startswith("Diff_") or c.startswith("Same_"):
            if pd.api.types.is_numeric_dtype(train_df[c]):
                feat_cols.append(c)

    # wywalamy stałe
    feat_cols = [c for c in feat_cols if train_df[c].nunique(dropna=True) > 1]
    return feat_cols

def clip_probs(p, eps=1e-6):
    return np.clip(p, eps, 1 - eps)

def score_predictions(y_true, p):
    p = clip_probs(p)
    return {
        "brier": brier_score_loss(y_true, p),
        "logloss": log_loss(y_true, p),
    }

m_features = get_feature_columns(m_train, m_valid, m_pred)
w_features = get_feature_columns(w_train, w_valid, w_pred)

print("M feature count:", len(m_features))
print("W feature count:", len(w_features))
print("\nTop M features:", m_features[:15])
print("\nTop W features:", w_features[:15])

M feature count: 85
W feature count: 75

Top M features: ['Diff_AstRateAllowedAvg', 'Diff_AstRateAvg', 'Diff_AvgAst', 'Diff_AvgBlk', 'Diff_AvgDR', 'Diff_AvgFGA', 'Diff_AvgFGA3', 'Diff_AvgFGM', 'Diff_AvgFGM3', 'Diff_AvgFTA', 'Diff_AvgFTM', 'Diff_AvgOR', 'Diff_AvgPF', 'Diff_AvgStl', 'Diff_AvgTO']

Top W features: ['Diff_AstRateAllowedAvg', 'Diff_AstRateAvg', 'Diff_AvgAst', 'Diff_AvgBlk', 'Diff_AvgDR', 'Diff_AvgFGA', 'Diff_AvgFGA3', 'Diff_AvgFGM', 'Diff_AvgFGM3', 'Diff_AvgFTA', 'Diff_AvgFTM', 'Diff_AvgOR', 'Diff_AvgPF', 'Diff_AvgStl', 'Diff_AvgTO']


In [3]:
def build_model_registry():
    models = {}

    # 1. logistic L2
    models["logreg_l2"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="liblinear",
            max_iter=5000,
            random_state=RANDOM_STATE
        ))
    ])

    # 2. logistic L1
    models["logreg_l1"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l1",
            C=0.5,
            solver="liblinear",
            max_iter=5000,
            random_state=RANDOM_STATE
        ))
    ])

    # 3. elastic net
    models["logreg_elastic"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="elasticnet",
            C=0.7,
            l1_ratio=0.5,
            solver="saga",
            max_iter=8000,
            random_state=RANDOM_STATE
        ))
    ])

    # 4. random forest
    models["rf"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=500,
            max_depth=6,
            min_samples_leaf=5,
            max_features="sqrt",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

    # 5. extra trees
    models["extra_trees"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=700,
            max_depth=6,
            min_samples_leaf=4,
            max_features="sqrt",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

    # 6. hist gradient boosting
    models["hist_gb"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            loss="log_loss",
            learning_rate=0.03,
            max_depth=3,
            max_iter=300,
            min_samples_leaf=10,
            l2_regularization=0.5,
            random_state=RANDOM_STATE
        ))
    ])

    # 7. XGBoost opcjonalnie
    try:
        from xgboost import XGBClassifier
        models["xgb"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                n_estimators=400,
                max_depth=3,
                learning_rate=0.03,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_alpha=0.0,
                reg_lambda=1.0,
                min_child_weight=3,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ])
    except Exception:
        pass

    # 8. LightGBM opcjonalnie
    try:
        from lightgbm import LGBMClassifier
        models["lgbm"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMClassifier(
                n_estimators=400,
                learning_rate=0.03,
                max_depth=3,
                num_leaves=15,
                subsample=0.85,
                colsample_bytree=0.85,
                min_child_samples=15,
                reg_alpha=0.0,
                reg_lambda=1.0,
                objective="binary",
                random_state=RANDOM_STATE,
                verbose=-1
            ))
        ])
    except Exception:
        pass

    # 9. CatBoost opcjonalnie
    try:
        from catboost import CatBoostClassifier
        models["catboost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", CatBoostClassifier(
                iterations=400,
                depth=4,
                learning_rate=0.03,
                loss_function="Logloss",
                eval_metric="Logloss",
                random_seed=RANDOM_STATE,
                verbose=False
            ))
        ])
    except Exception:
        pass

    return models

models = build_model_registry()
print("Loaded models:", list(models.keys()))

Loaded models: ['logreg_l2', 'logreg_l1', 'logreg_elastic', 'rf', 'extra_trees', 'hist_gb', 'xgb', 'lgbm', 'catboost']


In [5]:
from sklearn.base import clone

def evaluate_model_family(train_df, valid_df, feature_cols, model_registry, label):
    X_train = train_df[feature_cols].copy()
    y_train = train_df["Target"].astype(int).copy()

    X_valid = valid_df[feature_cols].copy()
    y_valid = valid_df["Target"].astype(int).copy()

    rows = []
    fitted_models = {}

    for model_name, model_template in model_registry.items():
        # świeża kopia modelu dla każdego fitu
        model = clone(model_template)

        model.fit(X_train, y_train)
        p_valid = model.predict_proba(X_valid)[:, 1]
        metrics = score_predictions(y_valid, p_valid)

        rows.append({
            "dataset": label,
            "model": model_name,
            "n_features": len(feature_cols),
            "brier_valid": metrics["brier"],
            "logloss_valid": metrics["logloss"],
        })

        fitted_models[model_name] = model

    results = pd.DataFrame(rows).sort_values(["brier_valid", "logloss_valid"]).reset_index(drop=True)
    return results, fitted_models

# osobne rejestry modeli dla M i W
models_m = build_model_registry()
models_w = build_model_registry()

m_results, m_fitted = evaluate_model_family(m_train, m_valid, m_features, models_m, "MEN")
w_results, w_fitted = evaluate_model_family(w_train, w_valid, w_features, models_w, "WOMEN")

print("=== MEN RESULTS ===")
display(m_results)

print("=== WOMEN RESULTS ===")
display(w_results)

=== MEN RESULTS ===


,dataset,model,n_features,brier_valid,logloss_valid
0,MEN,logreg_l2,85,0.154774,0.470032
1,MEN,logreg_elastic,85,0.155692,0.471290
2,MEN,logreg_l1,85,0.156184,0.472721
3,MEN,rf,85,0.160829,0.492622
4,MEN,extra_trees,85,0.170670,0.517093
5,MEN,catboost,85,0.172970,0.515778
6,MEN,hist_gb,85,0.186244,0.549843
7,MEN,xgb,85,0.190731,0.563893
8,MEN,lgbm,85,0.195426,0.576714


=== WOMEN RESULTS ===


,dataset,model,n_features,brier_valid,logloss_valid
0,WOMEN,hist_gb,75,0.105033,0.327025
1,WOMEN,xgb,75,0.105874,0.323618
2,WOMEN,lgbm,75,0.107534,0.329094
3,WOMEN,catboost,75,0.111366,0.342522
4,WOMEN,rf,75,0.111658,0.366929
5,WOMEN,logreg_l1,75,0.118212,0.361947
6,WOMEN,logreg_elastic,75,0.126907,0.378720
7,WOMEN,extra_trees,75,0.134671,0.433446
8,WOMEN,logreg_l2,75,0.136420,0.398242


In [6]:
def show_top_features(fitted_model, feature_cols, top_n=20):
    model = fitted_model.named_steps["model"]

    if hasattr(model, "coef_"):
        imp = pd.DataFrame({
            "feature": feature_cols,
            "coef": model.coef_[0]
        })
        imp["abs_coef"] = imp["coef"].abs()
        return imp.sort_values("abs_coef", ascending=False).head(top_n)

    if hasattr(model, "feature_importances_"):
        imp = pd.DataFrame({
            "feature": feature_cols,
            "importance": model.feature_importances_
        })
        return imp.sort_values("importance", ascending=False).head(top_n)

    return pd.DataFrame({"info": ["Brak natywnego importance dla tego modelu"]})

best_m_name = m_results.iloc[0]["model"]
best_w_name = w_results.iloc[0]["model"]

print("Best MEN model:", best_m_name)
display(show_top_features(m_fitted[best_m_name], m_features, top_n=20))

print("Best WOMEN model:", best_w_name)
display(show_top_features(w_fitted[best_w_name], w_features, top_n=20))

Best MEN model: logreg_l2


,feature,coef,abs_coef
12,Diff_AvgPF,-1.098141,1.098141
77,Diff_TOVForcedPctAvg,0.920106,0.920106
51,Diff_MasseyRankStd,-0.822404,0.822404
30,Diff_FTRAllowedAvg,0.814888,0.814888
52,Diff_MasseyRankWorst,0.620714,0.620714
75,Diff_SeedNum,0.575838,0.575838
31,Diff_FTRAvg,-0.513872,0.513872
35,Diff_HomeWins,-0.506630,0.506630
71,Diff_OppWinPctAvg,0.479911,0.479911
25,Diff_ConferenceGames,0.467280,0.467280


Best WOMEN model: hist_gb


,info
0,Brak natywnego importance dla tego modelu


In [7]:
def refit_best_and_predict(full_df, pred_df, feature_cols, fitted_model_template):
    X_full = full_df[feature_cols].copy()
    y_full = full_df["Target"].astype(int).copy()

    X_pred = pred_df[feature_cols].copy()

    fitted_model_template.fit(X_full, y_full)
    p_pred = fitted_model_template.predict_proba(X_pred)[:, 1]
    p_pred = clip_probs(p_pred)

    return pd.DataFrame({
        "ID": pred_df["ID"].values,
        "Pred": p_pred
    })

best_m_model = m_fitted[best_m_name]
best_w_model = w_fitted[best_w_name]

m_submission = refit_best_and_predict(m_full, m_pred, m_features, best_m_model)
w_submission = refit_best_and_predict(w_full, w_pred, w_features, best_w_model)

submission = (
    pd.concat([m_submission, w_submission], ignore_index=True)
      .sort_values("ID")
      .reset_index(drop=True)
)

out_path = DATA_DIR / "python_submission_best_single_model.csv"
submission.to_csv(out_path, index=False)

print("Saved:", out_path)
display(submission.head())
print("submission shape:", submission.shape)

Saved: data\interim\python_submission_best_single_model.csv


,ID,Pred
0,2026_1101_1102,0.126845
1,2026_1101_1103,0.039410
2,2026_1101_1104,0.020793
3,2026_1101_1105,0.700199
4,2026_1101_1106,0.570228


submission shape: (132133, 2)


In [8]:
from pathlib import Path
import itertools
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

RANDOM_STATE = 42
DATA_DIR = Path("data/interim")

def clip_probs(p, eps=1e-6):
    return np.clip(p, eps, 1 - eps)

def score_predictions(y_true, p):
    p = clip_probs(p)
    return {
        "brier": brier_score_loss(y_true, p),
        "logloss": log_loss(y_true, p),
    }

def get_top_corr_features(train_df, feature_cols, top_k=20):
    rows = []
    y = train_df["Target"].astype(int)

    for c in feature_cols:
        s = train_df[c]
        if s.notna().sum() < 10:
            continue
        if s.nunique(dropna=True) <= 1:
            continue
        corr = s.corr(y)
        if pd.notna(corr):
            rows.append((c, corr, abs(corr)))

    corr_df = pd.DataFrame(rows, columns=["feature", "corr", "abs_corr"])
    corr_df = corr_df.sort_values("abs_corr", ascending=False).reset_index(drop=True)
    return corr_df["feature"].head(top_k).tolist(), corr_df

def build_logreg_l2(C=1.0):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l2",
            C=C,
            solver="liblinear",
            max_iter=5000,
            random_state=RANDOM_STATE
        ))
    ])

def build_logreg_l1(C=0.5):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l1",
            C=C,
            solver="liblinear",
            max_iter=5000,
            random_state=RANDOM_STATE
        ))
    ])

def build_logreg_elastic(C=0.7, l1_ratio=0.5):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="elasticnet",
            C=C,
            l1_ratio=l1_ratio,
            solver="saga",
            max_iter=8000,
            random_state=RANDOM_STATE
        ))
    ])

def build_hist_gb(
    learning_rate=0.03,
    max_depth=3,
    max_iter=300,
    min_samples_leaf=10,
    l2_regularization=0.5
):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            loss="log_loss",
            learning_rate=learning_rate,
            max_depth=max_depth,
            max_iter=max_iter,
            min_samples_leaf=min_samples_leaf,
            l2_regularization=l2_regularization,
            random_state=RANDOM_STATE
        ))
    ])

def build_xgb(
    n_estimators=400,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    reg_lambda=1.0
):
    from xgboost import XGBClassifier
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            min_child_weight=min_child_weight,
            reg_alpha=0.0,
            reg_lambda=reg_lambda,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

def build_lgbm(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=3,
    num_leaves=15,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_samples=15,
    reg_lambda=1.0
):
    from lightgbm import LGBMClassifier
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LGBMClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            num_leaves=num_leaves,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            min_child_samples=min_child_samples,
            reg_alpha=0.0,
            reg_lambda=reg_lambda,
            objective="binary",
            random_state=RANDOM_STATE,
            verbose=-1
        ))
    ])

def manual_tune_model_grid(
    train_df,
    valid_df,
    pred_df,
    feature_cols,
    model_builder,
    param_grid,
    dataset_label,
    model_label
):
    X_train = train_df[feature_cols].copy()
    y_train = train_df["Target"].astype(int).copy()

    X_valid = valid_df[feature_cols].copy()
    y_valid = valid_df["Target"].astype(int).copy()

    rows = []
    fitted = {}

    for i, params in enumerate(param_grid, start=1):
        model = model_builder(**params)
        model.fit(X_train, y_train)

        p_valid = model.predict_proba(X_valid)[:, 1]
        scores = score_predictions(y_valid, p_valid)

        key = f"{model_label}__{i}"
        fitted[key] = {
            "model": model,
            "params": params,
            "features": feature_cols
        }

        rows.append({
            "dataset": dataset_label,
            "model_family": model_label,
            "run_id": key,
            "n_features": len(feature_cols),
            "params": params,
            "brier_valid": scores["brier"],
            "logloss_valid": scores["logloss"],
        })

    results = pd.DataFrame(rows).sort_values(["brier_valid", "logloss_valid"]).reset_index(drop=True)
    return results, fitted

def refit_and_predict(full_df, pred_df, feature_cols, model_builder, best_params):
    X_full = full_df[feature_cols].copy()
    y_full = full_df["Target"].astype(int).copy()
    X_pred = pred_df[feature_cols].copy()

    model = model_builder(**best_params)
    model.fit(X_full, y_full)

    p_pred = model.predict_proba(X_pred)[:, 1]
    p_pred = clip_probs(p_pred)

    return model, pd.DataFrame({
        "ID": pred_df["ID"].values,
        "Pred": p_pred
    })

In [9]:
# -----------------------------------------
# MEN: feature reduction by train correlation
# -----------------------------------------
m_top12, m_corr_df = get_top_corr_features(m_train, m_features, top_k=12)
m_top20, _ = get_top_corr_features(m_train, m_features, top_k=20)
m_top30, _ = get_top_corr_features(m_train, m_features, top_k=30)

print("MEN top 12 features:")
print(m_top12)

print("\nMEN top 20 features:")
print(m_top20[:20])

display(m_corr_df.head(25))

# -----------------------------------------
# MEN grids
# -----------------------------------------
m_logreg_l2_grid = [
    {"C": 0.10},
    {"C": 0.25},
    {"C": 0.50},
    {"C": 1.00},
    {"C": 2.00},
]

m_logreg_elastic_grid = [
    {"C": 0.20, "l1_ratio": 0.20},
    {"C": 0.50, "l1_ratio": 0.30},
    {"C": 0.70, "l1_ratio": 0.50},
    {"C": 1.00, "l1_ratio": 0.50},
    {"C": 1.50, "l1_ratio": 0.70},
]

m_hist_grid = [
    {"learning_rate": 0.02, "max_depth": 2, "max_iter": 250, "min_samples_leaf": 12, "l2_regularization": 0.5},
    {"learning_rate": 0.03, "max_depth": 2, "max_iter": 300, "min_samples_leaf": 10, "l2_regularization": 0.5},
    {"learning_rate": 0.03, "max_depth": 3, "max_iter": 250, "min_samples_leaf": 10, "l2_regularization": 1.0},
    {"learning_rate": 0.05, "max_depth": 2, "max_iter": 200, "min_samples_leaf": 8,  "l2_regularization": 1.0},
]

# -----------------------------------------
# MEN tune on top12
# -----------------------------------------
m_l2_top12_results, m_l2_top12_fit = manual_tune_model_grid(
    m_train, m_valid, m_pred, m_top12,
    build_logreg_l2, m_logreg_l2_grid,
    "MEN", "logreg_l2_top12"
)

m_elastic_top12_results, m_elastic_top12_fit = manual_tune_model_grid(
    m_train, m_valid, m_pred, m_top12,
    build_logreg_elastic, m_logreg_elastic_grid,
    "MEN", "logreg_elastic_top12"
)

m_hist_top12_results, m_hist_top12_fit = manual_tune_model_grid(
    m_train, m_valid, m_pred, m_top12,
    build_hist_gb, m_hist_grid,
    "MEN", "hist_gb_top12"
)

# -----------------------------------------
# MEN tune on top20
# -----------------------------------------
m_l2_top20_results, m_l2_top20_fit = manual_tune_model_grid(
    m_train, m_valid, m_pred, m_top20,
    build_logreg_l2, m_logreg_l2_grid,
    "MEN", "logreg_l2_top20"
)

m_elastic_top20_results, m_elastic_top20_fit = manual_tune_model_grid(
    m_train, m_valid, m_pred, m_top20,
    build_logreg_elastic, m_logreg_elastic_grid,
    "MEN", "logreg_elastic_top20"
)

m_hist_top20_results, m_hist_top20_fit = manual_tune_model_grid(
    m_train, m_valid, m_pred, m_top20,
    build_hist_gb, m_hist_grid,
    "MEN", "hist_gb_top20"
)

m_tuning_results = pd.concat([
    m_l2_top12_results,
    m_elastic_top12_results,
    m_hist_top12_results,
    m_l2_top20_results,
    m_elastic_top20_results,
    m_hist_top20_results,
], ignore_index=True).sort_values(["brier_valid", "logloss_valid"]).reset_index(drop=True)

print("=== MEN MINI TUNING RESULTS ===")
display(m_tuning_results)

MEN top 12 features:
['Diff_MasseyRankStd', 'Diff_SeedNum', 'Diff_MasseyRankWorst', 'Diff_MasseyRankMean', 'Diff_MasseyRankMedian', 'Diff_MasseySystemsCount', 'Diff_MasseyRankBest', 'Diff_MasseyRankBestInv', 'Diff_MarginAvg', 'Diff_NetRtgAvg', 'Diff_OppNetRtgAvg', 'Diff_MasseyRankMeanInv']

MEN top 20 features:
['Diff_MasseyRankStd', 'Diff_SeedNum', 'Diff_MasseyRankWorst', 'Diff_MasseyRankMean', 'Diff_MasseyRankMedian', 'Diff_MasseySystemsCount', 'Diff_MasseyRankBest', 'Diff_MasseyRankBestInv', 'Diff_MarginAvg', 'Diff_NetRtgAvg', 'Diff_OppNetRtgAvg', 'Diff_MasseyRankMeanInv', 'Diff_OppWinPctAvg', 'Diff_HomeWins', 'Diff_Wins', 'Diff_MasseyRankMedianInv', 'Diff_OffRtgAvg', 'Diff_AwayGames', 'Diff_HomeGames', 'Diff_WinPct']


,feature,corr,abs_corr
0,Diff_MasseyRankStd,-0.445157,0.445157
1,Diff_SeedNum,-0.444752,0.444752
2,Diff_MasseyRankWorst,-0.431622,0.431622
3,Diff_MasseyRankMean,-0.423979,0.423979
4,Diff_MasseyRankMedian,-0.423822,0.423822
5,Diff_MasseySystemsCount,0.419670,0.419670
6,Diff_MasseyRankBest,-0.376480,0.376480
7,Diff_MasseyRankBestInv,0.372370,0.372370
8,Diff_MarginAvg,0.361725,0.361725
9,Diff_NetRtgAvg,0.359745,0.359745


=== MEN MINI TUNING RESULTS ===


,dataset,model_family,run_id,n_features,params,brier_valid,logloss_valid
0,MEN,logreg_l2_top12,logreg_l2_top12__5,12,{'C': 2.0},0.150494,0.463883
1,MEN,logreg_l2_top12,logreg_l2_top12__4,12,{'C': 1.0},0.150503,0.464172
2,MEN,logreg_l2_top12,logreg_l2_top12__3,12,{'C': 0.5},0.150562,0.464610
3,MEN,logreg_elastic_top12,logreg_elastic_top12__5,12,"{'C': 1.5, 'l1_ratio': 0.7}",0.150633,0.464554
4,MEN,logreg_elastic_top12,logreg_elastic_top12__4,12,"{'C': 1.0, 'l1_ratio': 0.5}",0.150670,0.464748
5,MEN,logreg_l2_top12,logreg_l2_top12__2,12,{'C': 0.25},0.150720,0.465348
6,MEN,logreg_elastic_top12,logreg_elastic_top12__2,12,"{'C': 0.5, 'l1_ratio': 0.3}",0.150818,0.465378
7,MEN,logreg_elastic_top12,logreg_elastic_top12__3,12,"{'C': 0.7, 'l1_ratio': 0.5}",0.150825,0.465295
8,MEN,logreg_l2_top12,logreg_l2_top12__1,12,{'C': 0.1},0.151173,0.467108
9,MEN,logreg_elastic_top12,logreg_elastic_top12__1,12,"{'C': 0.2, 'l1_ratio': 0.2}",0.151257,0.467047


In [10]:
# -----------------------------------------
# WOMEN grids
# -----------------------------------------
w_hist_grid = [
    {"learning_rate": 0.02, "max_depth": 2, "max_iter": 250, "min_samples_leaf": 12, "l2_regularization": 0.5},
    {"learning_rate": 0.03, "max_depth": 2, "max_iter": 300, "min_samples_leaf": 10, "l2_regularization": 0.5},
    {"learning_rate": 0.03, "max_depth": 3, "max_iter": 300, "min_samples_leaf": 10, "l2_regularization": 0.5},
    {"learning_rate": 0.05, "max_depth": 3, "max_iter": 220, "min_samples_leaf": 8,  "l2_regularization": 1.0},
    {"learning_rate": 0.05, "max_depth": 2, "max_iter": 260, "min_samples_leaf": 8,  "l2_regularization": 1.0},
]

w_hist_results, w_hist_fit = manual_tune_model_grid(
    w_train, w_valid, w_pred, w_features,
    build_hist_gb, w_hist_grid,
    "WOMEN", "hist_gb"
)

display(w_hist_results)

# XGB optional
try:
    w_xgb_grid = [
        {"n_estimators": 250, "max_depth": 2, "learning_rate": 0.03, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_weight": 3, "reg_lambda": 1.0},
        {"n_estimators": 350, "max_depth": 2, "learning_rate": 0.03, "subsample": 0.90, "colsample_bytree": 0.85, "min_child_weight": 3, "reg_lambda": 1.0},
        {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.03, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_weight": 3, "reg_lambda": 1.0},
        {"n_estimators": 220, "max_depth": 2, "learning_rate": 0.05, "subsample": 0.90, "colsample_bytree": 0.90, "min_child_weight": 2, "reg_lambda": 1.0},
    ]

    w_xgb_results, w_xgb_fit = manual_tune_model_grid(
        w_train, w_valid, w_pred, w_features,
        build_xgb, w_xgb_grid,
        "WOMEN", "xgb"
    )
    display(w_xgb_results)
except Exception as e:
    w_xgb_results = pd.DataFrame()
    w_xgb_fit = {}
    print("XGB skipped:", e)

# LGBM optional
try:
    w_lgbm_grid = [
        {"n_estimators": 250, "learning_rate": 0.03, "max_depth": 2, "num_leaves": 7,  "subsample": 0.85, "colsample_bytree": 0.85, "min_child_samples": 18, "reg_lambda": 1.0},
        {"n_estimators": 350, "learning_rate": 0.03, "max_depth": 3, "num_leaves": 15, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_samples": 15, "reg_lambda": 1.0},
        {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 2, "num_leaves": 7,  "subsample": 0.90, "colsample_bytree": 0.90, "min_child_samples": 15, "reg_lambda": 1.0},
        {"n_estimators": 220, "learning_rate": 0.05, "max_depth": 3, "num_leaves": 15, "subsample": 0.85, "colsample_bytree": 0.90, "min_child_samples": 12, "reg_lambda": 1.5},
    ]

    w_lgbm_results, w_lgbm_fit = manual_tune_model_grid(
        w_train, w_valid, w_pred, w_features,
        build_lgbm, w_lgbm_grid,
        "WOMEN", "lgbm"
    )
    display(w_lgbm_results)
except Exception as e:
    w_lgbm_results = pd.DataFrame()
    w_lgbm_fit = {}
    print("LGBM skipped:", e)

w_tuning_results = pd.concat(
    [df for df in [w_hist_results, w_xgb_results, w_lgbm_results] if len(df) > 0],
    ignore_index=True
).sort_values(["brier_valid", "logloss_valid"]).reset_index(drop=True)

print("=== WOMEN MINI TUNING RESULTS ===")
display(w_tuning_results)

,dataset,model_family,run_id,n_features,params,brier_valid,logloss_valid
0,WOMEN,hist_gb,hist_gb__2,75,"{'learning_rate': 0.03, 'max_depth': 2, 'max_i...",0.104836,0.336308
1,WOMEN,hist_gb,hist_gb__3,75,"{'learning_rate': 0.03, 'max_depth': 3, 'max_i...",0.105033,0.327025
2,WOMEN,hist_gb,hist_gb__5,75,"{'learning_rate': 0.05, 'max_depth': 2, 'max_i...",0.105115,0.333108
3,WOMEN,hist_gb,hist_gb__1,75,"{'learning_rate': 0.02, 'max_depth': 2, 'max_i...",0.105802,0.348128
4,WOMEN,hist_gb,hist_gb__4,75,"{'learning_rate': 0.05, 'max_depth': 3, 'max_i...",0.108143,0.331793


,dataset,model_family,run_id,n_features,params,brier_valid,logloss_valid
0,WOMEN,xgb,xgb__2,75,"{'n_estimators': 350, 'max_depth': 2, 'learnin...",0.103103,0.328127
1,WOMEN,xgb,xgb__1,75,"{'n_estimators': 250, 'max_depth': 2, 'learnin...",0.103502,0.333525
2,WOMEN,xgb,xgb__3,75,"{'n_estimators': 300, 'max_depth': 3, 'learnin...",0.103902,0.323017
3,WOMEN,xgb,xgb__4,75,"{'n_estimators': 220, 'max_depth': 2, 'learnin...",0.105455,0.331570


,dataset,model_family,run_id,n_features,params,brier_valid,logloss_valid
0,WOMEN,lgbm,lgbm__4,75,"{'n_estimators': 220, 'learning_rate': 0.05, '...",0.105731,0.325321
1,WOMEN,lgbm,lgbm__2,75,"{'n_estimators': 350, 'learning_rate': 0.03, '...",0.107993,0.332083
2,WOMEN,lgbm,lgbm__1,75,"{'n_estimators': 250, 'learning_rate': 0.03, '...",0.108904,0.347439
3,WOMEN,lgbm,lgbm__3,75,"{'n_estimators': 300, 'learning_rate': 0.05, '...",0.109375,0.340031


=== WOMEN MINI TUNING RESULTS ===


,dataset,model_family,run_id,n_features,params,brier_valid,logloss_valid
0,WOMEN,xgb,xgb__2,75,"{'n_estimators': 350, 'max_depth': 2, 'learnin...",0.103103,0.328127
1,WOMEN,xgb,xgb__1,75,"{'n_estimators': 250, 'max_depth': 2, 'learnin...",0.103502,0.333525
2,WOMEN,xgb,xgb__3,75,"{'n_estimators': 300, 'max_depth': 3, 'learnin...",0.103902,0.323017
3,WOMEN,hist_gb,hist_gb__2,75,"{'learning_rate': 0.03, 'max_depth': 2, 'max_i...",0.104836,0.336308
4,WOMEN,hist_gb,hist_gb__3,75,"{'learning_rate': 0.03, 'max_depth': 3, 'max_i...",0.105033,0.327025
5,WOMEN,hist_gb,hist_gb__5,75,"{'learning_rate': 0.05, 'max_depth': 2, 'max_i...",0.105115,0.333108
6,WOMEN,xgb,xgb__4,75,"{'n_estimators': 220, 'max_depth': 2, 'learnin...",0.105455,0.331570
7,WOMEN,lgbm,lgbm__4,75,"{'n_estimators': 220, 'learning_rate': 0.05, '...",0.105731,0.325321
8,WOMEN,hist_gb,hist_gb__1,75,"{'learning_rate': 0.02, 'max_depth': 2, 'max_i...",0.105802,0.348128
9,WOMEN,lgbm,lgbm__2,75,"{'n_estimators': 350, 'learning_rate': 0.03, '...",0.107993,0.332083


In [11]:
# -----------------------------------------
# collect MEN fitted objects
# -----------------------------------------
m_all_fit = {}
m_all_fit.update(m_l2_top12_fit)
m_all_fit.update(m_elastic_top12_fit)
m_all_fit.update(m_hist_top12_fit)
m_all_fit.update(m_l2_top20_fit)
m_all_fit.update(m_elastic_top20_fit)
m_all_fit.update(m_hist_top20_fit)

# -----------------------------------------
# collect WOMEN fitted objects
# -----------------------------------------
w_all_fit = {}
w_all_fit.update(w_hist_fit)
w_all_fit.update(w_xgb_fit)
w_all_fit.update(w_lgbm_fit)

best_m_row = m_tuning_results.iloc[0]
best_w_row = w_tuning_results.iloc[0]

best_m_run = best_m_row["run_id"]
best_w_run = best_w_row["run_id"]

print("BEST MEN:")
display(best_m_row.to_frame().T)

print("BEST WOMEN:")
display(best_w_row.to_frame().T)

best_m_info = m_all_fit[best_m_run]
best_w_info = w_all_fit[best_w_run]

print("Best MEN params:", best_m_info["params"])
print("Best MEN feature count:", len(best_m_info["features"]))
print("Best MEN features:", best_m_info["features"])

print("\nBest WOMEN params:", best_w_info["params"])
print("Best WOMEN feature count:", len(best_w_info["features"]))
print("Best WOMEN features:", best_w_info["features"][:25], "...")

BEST MEN:


,dataset,model_family,run_id,n_features,params,brier_valid,logloss_valid
0,MEN,logreg_l2_top12,logreg_l2_top12__5,12,{'C': 2.0},0.150494,0.463883


BEST WOMEN:


,dataset,model_family,run_id,n_features,params,brier_valid,logloss_valid
0,WOMEN,xgb,xgb__2,75,"{'n_estimators': 350, 'max_depth': 2, 'learnin...",0.103103,0.328127


Best MEN params: {'C': 2.0}
Best MEN feature count: 12
Best MEN features: ['Diff_MasseyRankStd', 'Diff_SeedNum', 'Diff_MasseyRankWorst', 'Diff_MasseyRankMean', 'Diff_MasseyRankMedian', 'Diff_MasseySystemsCount', 'Diff_MasseyRankBest', 'Diff_MasseyRankBestInv', 'Diff_MarginAvg', 'Diff_NetRtgAvg', 'Diff_OppNetRtgAvg', 'Diff_MasseyRankMeanInv']

Best WOMEN params: {'n_estimators': 350, 'max_depth': 2, 'learning_rate': 0.03, 'subsample': 0.9, 'colsample_bytree': 0.85, 'min_child_weight': 3, 'reg_lambda': 1.0}
Best WOMEN feature count: 75
Best WOMEN features: ['Diff_AstRateAllowedAvg', 'Diff_AstRateAvg', 'Diff_AvgAst', 'Diff_AvgBlk', 'Diff_AvgDR', 'Diff_AvgFGA', 'Diff_AvgFGA3', 'Diff_AvgFGM', 'Diff_AvgFGM3', 'Diff_AvgFTA', 'Diff_AvgFTM', 'Diff_AvgOR', 'Diff_AvgPF', 'Diff_AvgStl', 'Diff_AvgTO', 'Diff_AwayGames', 'Diff_AwayWinPct', 'Diff_AwayWins', 'Diff_ConfTourneyConfsPlayed', 'Diff_ConfTourneyGames', 'Diff_ConfTourneyLastDay', 'Diff_ConfTourneyLosses', 'Diff_ConfTourneyUndefeated', 'Diff_C

In [12]:
# -----------------------------------------
# MEN refit on full train
# -----------------------------------------
if best_m_row["model_family"].startswith("logreg_l2"):
    m_builder = build_logreg_l2
elif best_m_row["model_family"].startswith("logreg_elastic"):
    m_builder = build_logreg_elastic
else:
    m_builder = build_hist_gb

m_best_model, m_sub_v2 = refit_and_predict(
    full_df=m_full,
    pred_df=m_pred,
    feature_cols=best_m_info["features"],
    model_builder=m_builder,
    best_params=best_m_info["params"]
)

# -----------------------------------------
# WOMEN refit on full train
# -----------------------------------------
if best_w_row["model_family"] == "hist_gb":
    w_builder = build_hist_gb
elif best_w_row["model_family"] == "xgb":
    w_builder = build_xgb
else:
    w_builder = build_lgbm

w_best_model, w_sub_v2 = refit_and_predict(
    full_df=w_full,
    pred_df=w_pred,
    feature_cols=best_w_info["features"],
    model_builder=w_builder,
    best_params=best_w_info["params"]
)

submission_v2 = (
    pd.concat([m_sub_v2, w_sub_v2], ignore_index=True)
      .sort_values("ID")
      .reset_index(drop=True)
)

out_path_v2 = DATA_DIR / "python_submission_manual_tuned_v2.csv"
submission_v2.to_csv(out_path_v2, index=False)

print("Saved:", out_path_v2)
display(submission_v2.head())
print("submission_v2 shape:", submission_v2.shape)

Saved: data\interim\python_submission_manual_tuned_v2.csv


,ID,Pred
0,2026_1101_1102,0.602113
1,2026_1101_1103,0.215156
2,2026_1101_1104,0.046490
3,2026_1101_1105,0.753934
4,2026_1101_1106,0.635583


submission_v2 shape: (132133, 2)


In [13]:
# -----------------------------------------
# optional blended submission
# -----------------------------------------

def predict_with_builder(full_df, pred_df, feature_cols, model_builder, params):
    _, sub = refit_and_predict(
        full_df=full_df,
        pred_df=pred_df,
        feature_cols=feature_cols,
        model_builder=model_builder,
        best_params=params
    )
    return sub

# MEN second model: benchmark logreg_l2 on full feature set
m_sub_bench = predict_with_builder(
    full_df=m_full,
    pred_df=m_pred,
    feature_cols=m_features,
    model_builder=build_logreg_l2,
    params={"C": 1.0}
)

# WOMEN second tuned model
best_w_row_2 = w_tuning_results.iloc[1]
best_w_info_2 = w_all_fit[best_w_row_2["run_id"]]

if best_w_row_2["model_family"] == "hist_gb":
    w_builder_2 = build_hist_gb
elif best_w_row_2["model_family"] == "xgb":
    w_builder_2 = build_xgb
else:
    w_builder_2 = build_lgbm

w_sub_v2_b = predict_with_builder(
    full_df=w_full,
    pred_df=w_pred,
    feature_cols=best_w_info_2["features"],
    model_builder=w_builder_2,
    params=best_w_info_2["params"]
)

# blend
m_blend = m_sub_v2.merge(m_sub_bench, on="ID", suffixes=("_a", "_b"))
m_blend["Pred"] = 0.7 * m_blend["Pred_a"] + 0.3 * m_blend["Pred_b"]
m_blend = m_blend[["ID", "Pred"]]

w_blend = w_sub_v2.merge(w_sub_v2_b, on="ID", suffixes=("_a", "_b"))
w_blend["Pred"] = 0.6 * w_blend["Pred_a"] + 0.4 * w_blend["Pred_b"]
w_blend = w_blend[["ID", "Pred"]]

submission_v2b = (
    pd.concat([m_blend, w_blend], ignore_index=True)
      .sort_values("ID")
      .reset_index(drop=True)
)

out_path_v2b = DATA_DIR / "python_submission_manual_tuned_blend_v2b.csv"
submission_v2b.to_csv(out_path_v2b, index=False)

print("Saved:", out_path_v2b)
display(submission_v2b.head())
print("submission_v2b shape:", submission_v2b.shape)

Saved: data\interim\python_submission_manual_tuned_blend_v2b.csv


,ID,Pred
0,2026_1101_1102,0.459532
1,2026_1101_1103,0.162432
2,2026_1101_1104,0.038781
3,2026_1101_1105,0.737813
4,2026_1101_1106,0.615976


submission_v2b shape: (132133, 2)
